# Position IDs 回归测试 & Tail Pruning 实验

两组对比（各 50 个 GSM8K sample，seed=42）：

| 测试 | 说明 |
|------|------|
| **baseline** | 标准 dual-cache 生成（无 position_ids） |
| **regression** | 传连续 `position_ids=[0,1,...,L-1]`，应与 baseline 完全一致 |
| **tail** | warm-up 阶段裁剪 suffix 至仅保留 tail token + `position_ids` |

## 1. 环境设置

In [ ]:
import os, sys, gc
import torch
import numpy as np

GPU_ID = '0'   # 改成 '1' 即可在第二张卡跑
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['HF_ALLOW_CODE_EVAL'] = '1'
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = 'true'

NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
os.chdir(os.path.join(NOTEBOOK_DIR, 'llada'))
print(f'Working dir: {os.getcwd()}')

if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

torch.cuda.empty_cache(); gc.collect()
if torch.cuda.is_available():
    print(f'GPU {GPU_ID}: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB')

## 2. 模型 & 数据加载

In [ ]:
from transformers import AutoTokenizer, AutoConfig
from model.modeling_llada import LLaDAModelLM
from datasets import load_dataset

MODEL_PATH = 'GSAI-ML/LLaDA-8B-Instruct'
MASK_ID = 126336

config = AutoConfig.from_pretrained(MODEL_PATH)
config.flash_attention = True
model = LLaDAModelLM.from_pretrained(
    MODEL_PATH, trust_remote_code=True, torch_dtype=torch.bfloat16, config=config,
).eval().to('cuda')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

gsm8k = load_dataset('gsm8k', 'main', split='test')
print(f'Model loaded. GSM8K test: {len(gsm8k)} samples')

## 3. 构建 prompt（5-shot，前 50 题）

In [ ]:
import re

FEW_SHOT_EXAMPLES = """Question: Jen and Tyler are gymnasts practicing flips. Jen is practicing the triple-flip while Tyler is practicing the double-flip. Jen did sixteen triple-flips during practice. Tyler flipped in the air half the number of times Jen did. How many double-flips did Tyler do?
Answer: Jen did 16 triple-flips, so she did 16 * 3 = <<16*3=48>>48 flips.
Tyler did half the number of flips, so he did 48 / 2 = <<48/2=24>>24 flips.
A double flip has two flips, so Tyler did 24 / 2 = <<24/2=12>>12 double-flips.
#### 12

Question: Four people in a law firm are planning a party. Mary will buy a platter of pasta for $20 and a loaf of bread for $2. Elle and Andrea will split the cost for buying 4 cans of soda which cost $1.50 each, and chicken wings for $10. Joe will buy a cake that costs $5. How much more will Mary spend than the rest of the firm put together?
Answer: Mary will spend $20 + $2 = $<<20+2=22>>22.
Elle and Andrea will spend $1.5 x 4 = $<<1.5*4=6>>6 for the soda.
Elle and Andrea will spend $6 + $10 = $<<6+10=16>>16 for the soda and chicken wings.
Elle, Andrea, and Joe together will spend $16 + $5 = $<<16+5=21>>21.
So, Mary will spend $22 - $21 = $<<22-21=1>>1 more than all of them combined.
#### 1

Question: A charcoal grill burns fifteen coals to ash every twenty minutes of grilling. The grill ran for long enough to burn three bags of coals. Each bag of coal contains 60 coals. How long did the grill run?
Answer: The grill burned 3 * 60 = <<3*60=180>>180 coals.
It takes 20 minutes to burn 15 coals, so the grill ran for 180 / 15 * 20 = <<180/15*20=240>>240 minutes.
#### 240

Question: A bear is preparing to hibernate for the winter and needs to gain 1000 pounds. At the end of summer, the bear feasts on berries and small woodland animals. During autumn, it devours acorns and salmon. It gained a fifth of the weight it needed from berries during summer, and during autumn, it gained twice that amount from acorns. Salmon made up half of the remaining weight it had needed to gain. How many pounds did it gain eating small animals?
Answer: The bear gained 1 / 5 * 1000 = <<1/5*1000=200>>200 pounds from berries.
It gained 2 * 200 = <<2*200=400>>400 pounds from acorns.
It still needed 1000 - 200 - 400 = <<1000-200-400=400>>400 pounds.
Thus, it gained 400 / 2 = <<400/2=200>>200 pounds from salmon.
Therefore, the bear gained 400 - 200 = <<400-200=200>>200 pounds from small animals.
#### 200

Question: Brendan can cut 8 yards of grass per day, he bought a lawnmower and it helped him to cut more yards by Fifty percent per day. How many yards will Brendan be able to cut after a week?
Answer: The additional yard Brendan can cut after buying the lawnmower is 8 x 0.50 = <<8*0.50=4>>4 yards.
So, the total yards he can cut with the lawnmower is 8 + 4 = <<8+4=12>>12.
Therefore, the total number of yards he can cut in a week is 12 x 7 = <<12*7=84>>84 yards.
#### 84"""


def build_prompt(question: str) -> torch.Tensor:
    text = FEW_SHOT_EXAMPLES + f'\n\nQuestion: {question}\nAnswer:'
    messages = [{'role': 'user', 'content': text}]
    formatted = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    ids = tokenizer(formatted)['input_ids']
    return torch.tensor(ids, dtype=torch.long, device='cuda').unsqueeze(0)


def extract_answer(text: str) -> str | None:
    m = re.search(r'####\s*(-?[\d,]+\.?\d*)', text)
    return m.group(1).replace(',', '').strip() if m else None


LIMIT = 50
prompts = [build_prompt(gsm8k[i]['question']) for i in range(LIMIT)]
ref_answers = [extract_answer(gsm8k[i]['answer']) for i in range(LIMIT)]
print(f'Built {LIMIT} prompts, first length: {prompts[0].shape[1]} tokens')

## 4. 生成参数

In [ ]:
GEN_LENGTH = 256
STEPS = 256
BLOCK_LENGTH = 32
THRESHOLD = 0.9

# 要跑哪些测试（可注释掉不需要的）
TESTS_TO_RUN = ['baseline', 'regression', 'tail']
print(f'Tests: {TESTS_TO_RUN}  |  {LIMIT} samples each')

## 5. 生成函数

简化版 dual-cache block-by-block（无 expand），三种模式：
- **baseline**: 标准 warm-up
- **regression**: warm-up 时传连续 position_ids（应与 baseline 完全一致）
- **tail**: warm-up 用裁剪输入（prefix + block + tail token）+ position_ids

In [ ]:
from generate import get_num_transfer_tokens, get_transfer_index, add_gumbel_noise


@torch.no_grad()
def generate_test(model, prompt, steps=256, gen_length=256, block_length=32,
                  temperature=0., threshold=0.9, mask_id=126336,
                  mode='baseline'):
    B = prompt.shape[0]
    Lp = int(prompt.shape[1])
    seq_len = Lp + gen_length
    assert gen_length % block_length == 0
    num_blocks = gen_length // block_length
    assert steps % num_blocks == 0
    steps_per_block = steps // num_blocks

    x = torch.full((B, seq_len), mask_id, dtype=torch.long, device=model.device)
    x[:, :Lp] = prompt
    nfe = 0

    for nb in range(num_blocks):
        s = Lp + nb * block_length
        e = s + block_length

        block_mask = (x[:, s:e] == mask_id)
        num_tt = get_num_transfer_tokens(block_mask, steps_per_block)

        # ---- Warm-up (differs by mode) ----
        if mode == 'tail':
            # Full forward → KV cache for refine
            out = model(x, use_cache=True)
            past_kv = out.past_key_values
            nfe += 1

            # Pruned forward → logits with tail context
            keep_idx = list(range(e))
            if e < seq_len:
                keep_idx.append(seq_len - 1)
            keep_t = torch.tensor(keep_idx, device=x.device, dtype=torch.long)
            x_pruned = x[:, keep_t]
            pos_ids = keep_t.unsqueeze(0)
            out_pruned = model(x_pruned, position_ids=pos_ids)
            nfe += 1
            warm_logits = out_pruned.logits[:, s:e, :]

        elif mode == 'regression':
            pos_ids = torch.arange(seq_len, device=x.device, dtype=torch.long).unsqueeze(0)
            out = model(x, use_cache=True, position_ids=pos_ids)
            past_kv = out.past_key_values
            nfe += 1
            warm_logits = out.logits[:, s:e, :]

        else:  # baseline
            out = model(x, use_cache=True)
            past_kv = out.past_key_values
            nfe += 1
            warm_logits = out.logits[:, s:e, :]

        # Step 0: transfer using warm-up logits (block portion)
        block_mask_0 = (x[:, s:e] == mask_id)
        q0 = None if threshold is not None else num_tt[:, 0]
        x0_blk, ti_blk = get_transfer_index(
            warm_logits, temperature, 'low_confidence',
            block_mask_0, x[:, s:e], q0, threshold,
            allow_fallback=True,
        )
        blk_new = torch.where(ti_blk, x0_blk, x[:, s:e])
        x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)

        # ---- Refine (identical for all modes) ----
        rp = torch.zeros_like(x, dtype=torch.bool)
        rp[:, s:e] = True

        for i in range(1, steps_per_block):
            if (x[:, s:e] == mask_id).sum() == 0:
                break

            logits_blk = model(
                x[:, s:e], past_key_values=past_kv,
                use_cache=True, replace_position=rp
            ).logits

            mask_blk = (x[:, s:e] == mask_id)
            qi = None if threshold is not None else num_tt[:, i]
            x0_blk, ti_blk = get_transfer_index(
                logits_blk, temperature, 'low_confidence',
                mask_blk, x[:, s:e], qi, threshold,
                allow_fallback=True,
            )
            blk_new = torch.where(ti_blk, x0_blk, x[:, s:e])
            x = torch.cat([x[:, :s], blk_new, x[:, e:]], dim=1)
            nfe += 1

    return x, nfe


print('generate_test() defined — modes: baseline / regression / tail')

## 6. 跑测试

In [ ]:
import time

results = {}  # mode → {acc, nfe_total, time, gen_texts, matches}

for mode in TESTS_TO_RUN:
    print(f'\n{"="*60}')
    print(f'  Mode: {mode}')
    print(f'{"="*60}')

    correct = 0
    total_nfe = 0
    gen_texts = []
    t0 = time.time()

    for i in range(LIMIT):
        prompt = prompts[i]
        x, nfe = generate_test(
            model, prompt,
            steps=STEPS, gen_length=GEN_LENGTH, block_length=BLOCK_LENGTH,
            temperature=0.0, threshold=THRESHOLD, mask_id=MASK_ID,
            mode=mode,
        )
        total_nfe += nfe

        gen_text = tokenizer.decode(x[0, prompt.shape[1]:], skip_special_tokens=True)
        for stop in ['Question:', '\n\nQuestion']:
            if stop in gen_text:
                gen_text = gen_text.split(stop)[0]
        gen_texts.append(gen_text)

        gen_ans = extract_answer(gen_text)
        is_correct = gen_ans is not None and ref_answers[i] is not None and gen_ans == ref_answers[i]
        if is_correct:
            correct += 1

        if (i + 1) % 10 == 0 or i == LIMIT - 1:
            elapsed = time.time() - t0
            print(f'  [{i+1}/{LIMIT}] NFE={nfe:3d}  acc={correct}/{i+1}  ({elapsed:.0f}s)')

    elapsed = time.time() - t0
    acc = correct / LIMIT
    results[mode] = {
        'acc': acc, 'nfe_total': total_nfe,
        'time': elapsed, 'gen_texts': gen_texts,
    }
    print(f'  → acc={acc:.2%}  NFE={total_nfe}  time={elapsed:.0f}s')

    torch.cuda.empty_cache(); gc.collect()

print(f'\n{"="*60}')
print('All done!')

## 7. 结果对比

In [ ]:
import pandas as pd

rows = []
for mode, r in results.items():
    rows.append({'mode': mode, 'accuracy': r['acc'], 'total_nfe': r['nfe_total'], 'time_sec': round(r['time'], 1)})
df = pd.DataFrame(rows)
display(df)

# --- Regression check: baseline vs regression should match exactly ---
if 'baseline' in results and 'regression' in results:
    bl = results['baseline']['gen_texts']
    rg = results['regression']['gen_texts']
    match_count = sum(1 for a, b in zip(bl, rg) if a == b)
    print(f'\n=== Regression check ===')
    print(f'  baseline vs regression 文本完全一致: {match_count}/{LIMIT}')
    if match_count < LIMIT:
        for idx, (a, b) in enumerate(zip(bl, rg)):
            if a != b:
                print(f'  Sample {idx} DIFFERS:')
                print(f'    baseline:   {a[:120]}...')
                print(f'    regression: {b[:120]}...')
                if idx >= 4:
                    print(f'    ... ({LIMIT - match_count - 5} more diffs)')
                    break

# --- Tail vs baseline quality delta ---
if 'baseline' in results and 'tail' in results:
    bl = results['baseline']['gen_texts']
    tl = results['tail']['gen_texts']
    match_count = sum(1 for a, b in zip(bl, tl) if a == b)
    print(f'\n=== Tail pruning quality ===')
    print(f'  baseline acc:  {results["baseline"]["acc"]:.2%}')
    print(f'  tail acc:      {results["tail"]["acc"]:.2%}')
    print(f'  delta:         {results["tail"]["acc"] - results["baseline"]["acc"]:+.2%}')
    print(f'  文本完全一致:  {match_count}/{LIMIT}')